# 07 · Interpretabilidad: importancia por impureza, por permutación y SHAP

**Módulo 4 · Sesión 11** — Boosting e interpretabilidad

## Objetivos

Los notebooks 04 y 06 eligieron un modelo por su AP. Ninguno respondió la pregunta que
cualquier enólogo haría: **¿qué hace que el modelo llame "buena" a una botella?** Este
notebook la responde con tres herramientas, y mide en qué se equivoca cada una:

1. La **importancia por impureza** (MDI) que traen gratis los bosques — y su sesgo hacia
   las variables con muchos valores distintos, medido plantando dos columnas de ruido.
2. La **importancia por permutación**, que mide sobre datos no vistos — y su punto ciego
   con variables correlacionadas, medido permutando en grupo.
3. **SHAP**: la contribución de cada variable a cada predicción, con la garantía de que las
   contribuciones **suman** exactamente la predicción. Global (qué variables importan y en
   qué dirección), local (por qué *esta* botella), y dependencia (cómo cambia el efecto de
   `alcohol` según la `density`).

La teoría está en `06-interpretabilidad.md`.

**Paquetes:** `pandas`, `numpy`, `matplotlib`, `scikit-learn`, `lightgbm`, `shap`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-4-clasificacion-ensambles/notebooks
    %pip install -q shap

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from lightgbm import LGBMClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold, train_test_split

warnings.filterwarnings("ignore")
SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. El modelo que se va a explicar

Mismos datos y partición de los notebooks 02, 04 y 06. Para la permutación y SHAP hace falta
un conjunto que el modelo **no haya visto** y que **no sea el de prueba** (que sigue
reservado): se usa el primer pliegue de la misma CV como validación, igual que en la
sección 3 del notebook 06.

Se explica el **LightGBM afinado** del notebook 06 (hiperparámetros copiados de su
resultado de Optuna). Extra-Trees tenía una AP algo mayor, pero `TreeExplainer` es exacto
y muy rápido para boosting, y —como se comprueba en la sección 3— las dos familias cuentan
la misma historia sobre las variables.

In [ ]:
vinos = pd.read_csv("../datos/wine-quality.csv")
vinos["tipo"] = (vinos["tipo"] == "tinto").astype(int)
vinos = vinos.drop_duplicates().reset_index(drop=True)
vinos["buena"] = (vinos["quality"] >= 7).astype(int)

X = vinos.drop(columns=["quality", "buena"])
y = vinos["buena"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
idx_tr, idx_va = next(cv.split(X_train, y_train))
X_tr, X_va = X_train.iloc[idx_tr], X_train.iloc[idx_va]
y_tr, y_va = y_train.iloc[idx_tr], y_train.iloc[idx_va]

PARAMETROS_LGBM = dict(
    n_estimators=615, learning_rate=0.093, num_leaves=38, min_child_samples=6,
    subsample=0.94, subsample_freq=1, colsample_bytree=0.77, reg_lambda=3.4,
)
lgbm = LGBMClassifier(**PARAMETROS_LGBM, random_state=SEMILLA, n_jobs=4, verbose=-1).fit(X_tr, y_tr)
print(f"AP de LightGBM sobre el pliegue de validación: {average_precision_score(y_va, lgbm.predict_proba(X_va)[:, 1]):.3f}")

## 2. Tres importancias para el mismo modelo

- **Impureza / ganancia** (MDI): cuánta reducción de pérdida acumuló cada variable en las
  particiones donde se usó. Se calcula durante el entrenamiento, sobre entrenamiento.
- **Permutación**: cuánto cae la AP de validación al **barajar** una columna (romper su
  relación con $y$ dejando todo lo demás intacto). Repetido 10 veces para promediar el
  azar del barajado.
- **SHAP** (media de $|\phi_j|$): cuánto mueve cada variable, en promedio, la predicción
  de cada vino respecto a la predicción base.

In [ ]:
mdi = pd.Series(lgbm.booster_.feature_importance(importance_type="gain"), index=X.columns)
mdi = mdi / mdi.sum()

perm = permutation_importance(lgbm, X_va, y_va, scoring="average_precision", n_repeats=10, random_state=SEMILLA)
perm_media = pd.Series(perm.importances_mean, index=X.columns)

explicador = shap.TreeExplainer(lgbm)
explicacion = explicador(X_va)  # objeto Explanation: valores SHAP en escala de log-momios
shap_media = pd.Series(np.abs(explicacion.values).mean(axis=0), index=X.columns)

importancias = pd.DataFrame({"ganancia (MDI)": mdi, "permutación (ΔAP)": perm_media, "SHAP (media |φ|)": shap_media})
importancias["rango MDI"] = importancias["ganancia (MDI)"].rank(ascending=False).astype(int)
importancias["rango perm."] = importancias["permutación (ΔAP)"].rank(ascending=False).astype(int)
importancias["rango SHAP"] = importancias["SHAP (media |φ|)"].rank(ascending=False).astype(int)
print(importancias.sort_values("SHAP (media |φ|)", ascending=False).round(3).to_string())

Las tres coinciden en lo grande: `alcohol` es, con diferencia, la variable principal, y
`tipo` no aporta **nada**. Lo segundo merece una pausa: `tipo` correlaciona 0.65 con
`volatile_acidity` y −0.69 con `total_sulfur_dioxide` — la química ya dice si un vino es
tinto o blanco, y el modelo no necesita la etiqueta. **Importancia cero no significa
"irrelevante para el problema", significa "redundante para este modelo"**: si se quitaran
las variables de azufre, `tipo` subiría.

Y discrepan en el medio de la tabla. `chlorides`, `ph`, `citric_acid` y `fixed_acidity`
tienen una ganancia parecida a la de `sulphates` (0.06–0.08), pero una permutación tres o
cuatro veces menor (≈0.015): el modelo las **usó** al entrenar, pero barajarlas apenas le
cuesta. Y `residual_sugar` va al revés: décima en ganancia, tercera en permutación. Las
secciones 3 y 4 explican por qué las dos medidas pueden decir cosas distintas.

## 3. El sesgo de la MDI, medido con ruido

Se añaden dos columnas que no tienen ninguna relación con la calidad: `ruido_continuo`
(gaussiano, 4256 valores distintos) y `ruido_binario` (0/1). Se ajusta un Random Forest —el
modelo cuya MDI todo el mundo grafica— y se comparan MDI y permutación.

In [ ]:
X_tr_ruido = X_tr.copy()
X_tr_ruido["ruido_continuo"] = rng.normal(size=len(X_tr))
X_tr_ruido["ruido_binario"] = rng.integers(0, 2, size=len(X_tr))
X_va_ruido = X_va.copy()
X_va_ruido["ruido_continuo"] = rng.normal(size=len(X_va))
X_va_ruido["ruido_binario"] = rng.integers(0, 2, size=len(X_va))

rf = RandomForestClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1).fit(X_tr_ruido, y_tr)
perm_rf = permutation_importance(rf, X_va_ruido, y_va, scoring="average_precision", n_repeats=10, random_state=SEMILLA)

tabla_ruido = pd.DataFrame(
    {"MDI (Random Forest)": rf.feature_importances_, "permutación (ΔAP)": perm_rf.importances_mean},
    index=X_tr_ruido.columns,
).sort_values("MDI (Random Forest)", ascending=False)
tabla_ruido["rango MDI"] = np.arange(1, len(tabla_ruido) + 1)
print(tabla_ruido.round(4).to_string())

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
for eje, col in zip(ejes, ["MDI (Random Forest)", "permutación (ΔAP)"]):
    serie = tabla_ruido[col].sort_values()
    colores = ["C3" if "ruido" in v else "C0" for v in serie.index]
    eje.barh(serie.index, serie.values, color=colores)
    eje.set_title(col)
plt.tight_layout()
plt.show()

`ruido_continuo` —una columna de números aleatorios— aparece en la MDI **por encima de
cuatro variables reales** (`free_sulfur_dioxide`, `ph`, `citric_acid` y `fixed_acidity`),
con una importancia 6 veces mayor que la de `ruido_binario`, que es exactamente igual de
inútil. La diferencia entre ambas es solo el
número de umbrales candidatos: con 4256 valores distintos siempre hay algún corte que
reduce la impureza por azar, y con árboles sin podar esos cortes se usan. La permutación,
medida sobre datos no vistos, les da a las dos un valor **indistinguible de cero** (o
negativo: barajarlas a veces *mejora* la AP, que es la firma del ruido).

Esta es la explicación de lo que `04-pipeline-caracteristicas-aplicado.ipynb` (módulo 2)
vio sin poder explicar: la MDI ponía `age` (continua) por encima del sexo (binario) en el
Titanic. Y es la razón por la que `fixed_acidity` y `citric_acid` tenían ganancia sin
tener permutación en la sección 2.

## 4. El punto ciego de la permutación: variables correlacionadas

Permutar una columna rompe su relación con $y$… pero también con las demás columnas. Si
`density` está muy correlacionada con `alcohol` (−0.67) y con `residual_sugar` (0.52), al
barajar `density` el modelo puede seguir "leyendo" casi la misma información en sus
compañeras, y la caída de AP subestima lo que `density` aporta. Además, las filas
barajadas son combinaciones **imposibles** (densidad de un vino dulce con el alcohol de
uno seco) sobre las que el modelo nunca aprendió.

La comprobación: permutar las tres **juntas** (manteniendo sus filas alineadas) y comparar
con la suma de las tres permutaciones individuales. Si el problema es grave, el grupo
valdrá bastante más que la suma.

In [ ]:
def ap_permutando(modelo, X, y, columnas, repeticiones=10):
    caidas = []
    base = average_precision_score(y, modelo.predict_proba(X)[:, 1])
    for r in range(repeticiones):
        X_perm = X.copy()
        orden = np.random.default_rng(SEMILLA + r).permutation(len(X))
        X_perm[columnas] = X[columnas].to_numpy()[orden]  # mismo orden para todas: se preserva su relación mutua
        caidas.append(base - average_precision_score(y, modelo.predict_proba(X_perm)[:, 1]))
    return np.mean(caidas)


grupo = ["density", "alcohol", "residual_sugar"]
individuales = {c: ap_permutando(lgbm, X_va, y_va, [c]) for c in grupo}
conjunta = ap_permutando(lgbm, X_va, y_va, grupo)

print("Caída de AP al permutar (LightGBM, validación):")
for c, v in individuales.items():
    print(f"  {c:15s} sola: {v:.3f}")
print(f"  suma de las tres individuales: {sum(individuales.values()):.3f}")
print(f"  las tres juntas:               {conjunta:.3f}")

Medido, el problema aquí es **pequeño**: el grupo vale 0.217 y la suma 0.208, un 4 % de
información compartida que ninguna permutación individual atribuye. Con correlaciones de
0.5–0.7 el modelo sigue necesitando a cada variable por separado. El punto ciego aparece de
verdad cuando la correlación es casi perfecta — y se puede provocar para verlo: se añade
una copia de `alcohol` con un ruido minúsculo y se reajusta el modelo.

In [ ]:
X_tr_copia = X_tr.copy()
X_tr_copia["alcohol_copia"] = X_tr["alcohol"] + rng.normal(0, 0.01, len(X_tr))
X_va_copia = X_va.copy()
X_va_copia["alcohol_copia"] = X_va["alcohol"] + rng.normal(0, 0.01, len(X_va))

lgbm_copia = LGBMClassifier(**PARAMETROS_LGBM, random_state=SEMILLA, n_jobs=4, verbose=-1).fit(X_tr_copia, y_tr)
print(f"AP con la copia: {average_precision_score(y_va, lgbm_copia.predict_proba(X_va_copia)[:, 1]):.3f} (sin ella: {average_precision_score(y_va, lgbm.predict_proba(X_va)[:, 1]):.3f})")
print("Caída de AP al permutar:")
print(f"  alcohol sola:              {ap_permutando(lgbm_copia, X_va_copia, y_va, ['alcohol']):.3f}   (sin la copia era {individuales['alcohol']:.3f})")
print(f"  alcohol_copia sola:        {ap_permutando(lgbm_copia, X_va_copia, y_va, ['alcohol_copia']):.3f}")
print(f"  las dos juntas:            {ap_permutando(lgbm_copia, X_va_copia, y_va, ['alcohol', 'alcohol_copia']):.3f}")

Con una copia casi exacta, la importancia por permutación de `alcohol` **se desploma**: el
modelo repartió sus particiones entre las dos columnas y, al barajar una, lee la otra. La
variable más importante del problema parece prescindible. Permutadas juntas, recuperan el
valor original. La permutación individual no inventa importancia, como la MDI, pero la
**reparte mal** entre variables casi redundantes; con grupos de variables que se sabe
correlacionadas, permutar por grupo es la lectura honesta — y la MDI, dicho sea de paso,
tiene el mismo problema (la ganancia también se reparte).

## 5. SHAP: contribuciones que suman la predicción

Los valores de Shapley reparten la diferencia entre la predicción de un vino y la
predicción base (el log-momio promedio) entre sus variables, con una propiedad que las
otras dos importancias no tienen: **las contribuciones suman exactamente la predicción**.
Para árboles, `TreeExplainer` los calcula de forma exacta y rápida.

In [ ]:
log_momios = lgbm.predict(X_va, raw_score=True)
reconstruido = explicacion.base_values + explicacion.values.sum(axis=1)
print(f"Valor base (log-momio promedio): {explicacion.base_values[0]:.3f}  →  P = {1 / (1 + np.exp(-explicacion.base_values[0])):.3f}")
print(f"(no es la prevalencia, 0.19: es el promedio de log-momios, y el modelo es muy confiado en los negativos)")
print(f"Máxima diferencia entre base + suma de SHAP y el log-momio del modelo: {np.abs(reconstruido - log_momios).max():.2e}")

### Global: qué variables y en qué dirección

El gráfico de enjambre (*beeswarm*) muestra, para cada variable, un punto por vino: su
posición horizontal es la contribución $\phi_j$ (en log-momios), y el color, el valor de
la variable. Es la importancia global **con signo**: no solo cuánto importa `alcohol`,
sino que más alcohol empuja hacia "buena".

In [ ]:
shap.plots.beeswarm(explicacion, max_display=12, show=False)
plt.title("SHAP sobre el pliegue de validación (851 vinos)")
plt.tight_layout()
plt.show()

Lectura: `alcohol` alto (rojo) empuja fuerte hacia "buena"; `volatile_acidity` alta y
`density` alta empujan hacia "no buena"; `sulphates` altos ayudan. Y hay asimetría: los
valores altos de `alcohol` empujan más lejos hacia la derecha que los bajos hacia la
izquierda — el efecto no es lineal, cosa que un coeficiente logístico no podría mostrar.

### Dependencia: el efecto de `alcohol` según la `density`

La contribución de una variable no es constante: depende de las demás. El gráfico de
dependencia muestra $\phi_{\text{alcohol}}$ contra el valor de `alcohol`, coloreado por la
variable con la que más interactúa.

In [ ]:
shap.plots.scatter(explicacion[:, "alcohol"], color=explicacion[:, "density"], show=False)
plt.title("Contribución SHAP de alcohol, coloreada por density")
plt.tight_layout()
plt.show()

La contribución de `alcohol` cruza el cero cerca de 10.7 %: por debajo penaliza, por
encima premia, y entre 9.5 y 13 % lo hace de forma casi lineal en log-momios — con un
aplanamiento por debajo de 9.5 %, donde ya no hay nada más que penalizar. Y para el mismo
alcohol, los vinos más densos (rojos) reciben una contribución menor que los ligeros
(azules): la interacción `alcohol × density` —que un modelo lineal necesitaría como término
explícito— está dentro de los árboles.

### Comparación con la dependencia parcial

El gráfico de **dependencia parcial** (PDP) responde una pregunta parecida de otra forma:
fija `alcohol` en cada valor de una malla para *todos* los vinos y promedia las
predicciones. Es más simple, pero promedia sobre combinaciones imposibles (todos los vinos
con 14 % de alcohol, incluidos los densos y dulces) y no muestra la interacción.

In [ ]:
fig, eje = plt.subplots(figsize=(6.5, 4))
PartialDependenceDisplay.from_estimator(lgbm, X_va, ["alcohol"], kind="both", subsample=100, random_state=SEMILLA, ax=eje)
eje.set_title("Dependencia parcial (línea gruesa) y curvas individuales (ICE) para alcohol")
plt.tight_layout()
plt.show()

La forma general coincide con la de SHAP —la probabilidad promedio empieza a subir cerca
de 10.5 %—, pero las curvas individuales (ICE) muestran lo que el promedio esconde: para
la mayoría de los vinos, subir el alcohol apenas mueve la probabilidad (siguen pegados a
cero), y para una minoría la dispara hasta casi 1. El promedio de 0.3 no describe a casi
ningún vino. Cuando las curvas ICE se abren en abanico así, hay interacciones fuertes, y
SHAP es la herramienta para verlas.

### Local: por qué esta botella

Dos vinos del pliegue de validación: el positivo con mayor probabilidad predicha (un
acierto claro) y el positivo con **menor** probabilidad (un falso negativo). El gráfico de
cascada muestra cómo cada variable mueve el log-momio desde el valor base hasta la
predicción.

In [ ]:
p_va = lgbm.predict_proba(X_va)[:, 1]
positivos = np.where(y_va.to_numpy() == 1)[0]
i_acierto = positivos[np.argmax(p_va[positivos])]
i_fallo = positivos[np.argmin(p_va[positivos])]

for i, etiqueta in [(i_acierto, "verdadero positivo más claro"), (i_fallo, "falso negativo más claro")]:
    print(f"\n{etiqueta}: P(buena) = {p_va[i]:.3f}   alcohol = {X_va.iloc[i]['alcohol']:.1f}   "
          f"density = {X_va.iloc[i]['density']:.4f}   volatile_acidity = {X_va.iloc[i]['volatile_acidity']:.2f}")
    shap.plots.waterfall(explicacion[i], max_display=8, show=False)
    plt.title(etiqueta)
    plt.tight_layout()
    plt.show()

En el acierto, casi todas las variables empujan en la misma dirección y `alcohol` lidera.
En el falso negativo, el vino es bueno según los catadores pero su química —9.3 % de
alcohol, cloruros altos— es la de un vino corriente: `alcohol` resta 1.7 log-momios y
`chlorides` otro 1.0, y el modelo no tiene forma de saber que los catadores opinaron
distinto. Una explicación local no arregla el error, pero dice **qué** tendría que ser
distinto para que la predicción cambiara — que es lo que un usuario del modelo necesita.

## 6. La misma historia con Extra-Trees

Para cerrar la promesa de la sección 1: la importancia por permutación del Extra-Trees del
notebook 06 (el modelo con mejor AP) sobre el mismo pliegue de validación.

In [ ]:
extra_trees = ExtraTreesClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1).fit(X_tr, y_tr)
perm_et = permutation_importance(extra_trees, X_va, y_va, scoring="average_precision", n_repeats=10, random_state=SEMILLA)
comparacion_modelos = pd.DataFrame(
    {"LightGBM (permutación)": perm_media, "Extra-Trees (permutación)": pd.Series(perm_et.importances_mean, index=X.columns)}
).sort_values("LightGBM (permutación)", ascending=False)
print(comparacion_modelos.round(3).to_string())
print(f"\nCorrelación de rangos entre ambos modelos: {comparacion_modelos.corr(method='spearman').iloc[0, 1]:.2f}")

## Resumen

| Herramienta | Qué mide | Dónde se equivoca (medido) |
|---|---|---|
| MDI / ganancia | Cuánto usó el modelo cada variable al entrenar | Una columna de ruido continuo queda por encima de tres variables reales y 6–7× por encima del ruido binario |
| Permutación | Cuánto cae la métrica de validación al barajar la variable | Reparte mal entre variables casi redundantes: con una copia de `alcohol`, su importancia se desploma; con correlaciones de 0.5–0.7, el efecto medido es de solo un 4 % |
| SHAP | Contribución de cada variable a cada predicción; suma exacta | Costoso fuera de los árboles; sus gráficos exigen leerse con cuidado (log-momios, no probabilidades) |

Y la respuesta al enólogo: el modelo llama "buena" a una botella sobre todo por su
**alcohol** (a partir de ≈10.7 %, y más cuanto menos densa), penaliza la **acidez volátil**,
la **densidad** y los **cloruros**, y premia los **sulfatos**; el tipo de vino no le hace
falta porque la química ya lo dice. `06-interpretabilidad.md` discute qué de esto es *explicación del modelo* y qué
sería *causalidad* — que no es lo mismo.